# Lab 06 — Quotas & cost (Quota / Overview panes)

**Control Plane panes:** _Quota_ and _Overview_

**Zava context:** the CFO wants to know two things every Monday:

1. **Are we headroom-safe?** — Are Zava's model deployments approaching their TPM/RPM caps? Which region should the next agent land in?
2. **Are we cost-safe?** — Which agents / models / projects burned the most spend last week?

This lab shows both angles via the management-plane SDKs and points at the portal panes that surface the same data.

> References:
> - [Optimize cost & performance](https://learn.microsoft.com/en-us/azure/foundry/control-plane/how-to-optimize-cost-performance)
> - [Enforce model limits](https://learn.microsoft.com/en-us/azure/foundry/control-plane/how-to-enforce-limits-models)

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from rich.console import Console
from rich.table import Table

# Reload the shared lab settings so stale notebook variables cannot select another subscription.
load_dotenv(Path.cwd().parent / ".env", override=True)

# DefaultAzureCredential reuses the active Azure CLI sign-in without embedding credentials.
credential = DefaultAzureCredential()
console = Console()

# The management client is subscription-scoped; later queries narrow results to this resource group.
SUB_ID = os.environ["AZURE_SUBSCRIPTION_ID"]
RG = os.environ["AZURE_RESOURCE_GROUP"]
cs = CognitiveServicesManagementClient(credential, SUB_ID)

## 1. Model deployment quota utilization

Each deployment has a **capacity** (TPM in thousands). We list every deployment on every Foundry-flavored account in the RG and print its capacity.

In [ ]:
table = Table(title=f"Deployments in RG '{RG}'")
for column_name in ("Account", "Deployment", "Model", "SKU", "Capacity (TPM k)"):
    table.add_column(column_name)

# A resource group can contain unrelated Cognitive Services accounts; keep only model hosts.
for account in cs.accounts.list_by_resource_group(RG):
    account_kind = (account.kind or "").lower()
    if account_kind not in {"aiservices", "openai"}:
        continue

    for deployment in cs.deployments.list(RG, account.name):
        # Management-plane responses can omit model or SKU data during provisioning.
        model = deployment.properties.model if deployment.properties else None
        model_name = getattr(model, "name", "") if model else ""
        sku_name = deployment.sku.name if deployment.sku else ""
        capacity = deployment.sku.capacity if deployment.sku else ""

        # For token-based model SKUs, capacity is conventionally expressed in thousands of TPM.
        table.add_row(
            account.name,
            deployment.name,
            model_name,
            sku_name,
            str(capacity),
        )

console.print(table)

## 2. Subscription-level model quota (per region, per model)

The **usages** API on `CognitiveServicesManagementClient` returns your remaining quota per region — this is what the Control Plane Quota pane visualises.

In [ ]:
# Quota is reported per region and model family, so query likely landing regions explicitly.
REGIONS = ["eastus", "eastus2", "westus", "westus3", "swedencentral", "francecentral"]

table = Table(title="Cognitive Services usages by region")
for column_name in ("Region", "Name", "Current", "Limit", "% used"):
    table.add_column(column_name)

for region in REGIONS:
    try:
        usages = list(cs.usages.list(location=region))
    except Exception as error:
        # Continue so one unsupported or inaccessible region does not hide all quota data.
        console.print(f"[yellow]{region}: {error}[/yellow]")
        continue

    for usage in usages:
        quota_name = usage.name.value if usage.name else ""
        current = usage.current_value or 0
        limit = usage.limit or 0

        # A zero limit is not actionable capacity and cannot produce a utilization percentage.
        if limit == 0:
            continue

        percent_used = (current / limit) * 100
        # Suppress untouched quota families to keep the table focused on active allocations.
        if current == 0:
            continue

        table.add_row(
            region,
            quota_name[:60],
            f"{current:.0f}",
            f"{limit:.0f}",
            f"{percent_used:.1f}%",
        )

console.print(table)

## 3. Cost by resource — last 7 days

Ask the Cost Management API to break down Foundry spend by resource for the past week. Requires the `Cost Management Reader` role on the subscription.

In [ ]:
import time
from datetime import datetime, timedelta, timezone
from azure.core.exceptions import HttpResponseError
from azure.mgmt.costmanagement import CostManagementClient
from azure.mgmt.costmanagement.models import (
    QueryAggregation,
    QueryDataset,
    QueryDefinition,
    QueryGrouping,
    QueryTimePeriod,
)

cost = CostManagementClient(credential)
scope = f"/subscriptions/{SUB_ID}"

# Cost Management expects timezone-aware UTC boundaries for a custom reporting window.
end = datetime.now(timezone.utc)
start = end - timedelta(days=7)

definition = QueryDefinition(
    type="Usage",
    timeframe="Custom",
    time_period=QueryTimePeriod(from_property=start, to=end),
    dataset=QueryDataset(
        # Aggregate the whole period server-side; daily rows are unnecessary for this top-N view.
        aggregation={"totalCost": QueryAggregation(name="Cost", function="Sum")},
        grouping=[
            QueryGrouping(type="Dimension", name="ResourceId"),
            QueryGrouping(type="Dimension", name="MeterCategory"),
        ],
    ),
)

# Cost Management can transiently throttle subscription queries, so retry only HTTP 429.
for attempt in range(1, 5):
    try:
        response = cost.query.usage(scope=scope, parameters=definition)
        break
    except HttpResponseError as error:
        if error.status_code != 429 or attempt == 4:
            raise
        retry_after = int(error.response.headers.get("Retry-After", 5 * attempt))
        console.print(f"[yellow]Cost query throttled; retrying in {retry_after}s...[/yellow]")
        time.sleep(retry_after)

# Resolve fields by their API-provided names instead of relying on positional ordering.
column_index = {column.name: index for index, column in enumerate(response.columns or [])}
required_columns = {"Cost", "ResourceId", "MeterCategory"}
missing_columns = required_columns - column_index.keys()
if missing_columns:
    raise ValueError(f"Cost response omitted columns: {sorted(missing_columns)}")

totals = {}
for row in response.rows or []:
    amount = row[column_index["Cost"]] or 0
    resource_id = row[column_index["ResourceId"]] or "(unattributed)"
    category = row[column_index["MeterCategory"]] or ""

    # Meter categories vary by offer; retain both Cognitive Services and AI categories.
    if "Cognitive" in category or "AI" in category:
        totals[resource_id] = totals.get(resource_id, 0) + amount

top_resources = sorted(totals.items(), key=lambda item: -item[1])[:10]
currency_column = column_index.get("Currency")
currency = (response.rows[0][currency_column] if response.rows and currency_column is not None else "")

table = Table(title="Top Foundry / Cognitive resource cost - last 7 days")
table.add_column("Resource")
table.add_column(f"Cost ({currency})" if currency else "Cost")
for resource_id, amount in top_resources:
    table.add_row(resource_id.split("/")[-1], f"{amount:,.2f}")
console.print(table)

## 4. Attribute cost back to Zava agents

Cost Management can only slice by resource / meter — it doesn't know about agents. To attribute cost per agent, join **traces** (Lab 04, token counts per agent) with **model pricing** (from the Foundry pricing page or `/openai/deployments/{name}/model` metadata).

```kql
// In App Insights → Logs
dependencies
| where timestamp > ago(7d)
| extend agent = tostring(customDimensions["service.name"])
| extend prompt = toint(customDimensions["gen_ai.usage.prompt_tokens"])
| extend completion = toint(customDimensions["gen_ai.usage.completion_tokens"])
| summarize prompt_tokens=sum(prompt), completion_tokens=sum(completion) by agent
```

Multiply by the per-1K-token price of the underlying model deployment to get $ / agent / week.

## 5. Set enforceable limits from the portal

Once you can see the numbers, enforce them:

> **Foundry portal → Operate → Admin → Allowed models & limits**

You can:

- Restrict which models developers may deploy (e.g. block preview models in prod projects).
- Cap capacity per deployment.
- Require approval for capacity increases above a threshold.

And for cost specifically:

> **Foundry portal → Operate → Overview → Cost signals** (or Azure Portal → Cost Management → Budgets)

Set a **Budget** on the Foundry resource group with an action group that emails the Zava platform lead when spend > 80% of monthly cap.

## Done

You've walked the entire Control Plane loop for the Zava fleet:

```
Setup → Inventory → Guardrails → Evaluations → Tracing → Red-team → Quota/Cost
```

This is the same loop the Zava platform team would automate in CI/CD once they're happy with the manual runs.